In [4]:
import pandas as pd
df = pd.read_csv("../data/cleaned_train.tsv", sep="\t")
df_sample = df.sample(n=2000, random_state=42)
df_sample.to_csv("../data/sample_dedup_test.tsv", sep="\t", index=False)


/tmp/ipykernel_2994582/3424191708.py:2: DtypeWarning: Columns (1,2,3,6,8,9,10,12,13,14,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/cleaned_train.tsv", sep="\t")


In [42]:
import os
import time
import argparse
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz
from joblib import Parallel, delayed
import hashlib
import pickle
import psutil
def compare_block(texts, idxs, threshold, non_missing_counts, log_matches=False):
    local_to_drop = set()
    for i in range(len(texts)):
        if idxs[i] in local_to_drop:
            continue
        for j in range(i + 1, len(texts)):
            if idxs[j] in local_to_drop:
                continue
            score = fuzz.ratio(texts[i], texts[j])
            threshold = threshold * 100
            if score >= threshold:
                if log_matches:
                    print(f"\n[Score {score}]")
                    print(f"A: {texts[i]}")
                    print(f"B: {texts[j]}")

    return local_to_drop


In [43]:
from dedup_data import load_hash_cache, save_hash_cache, normalize_text, hash_text
def block_and_deduplicate(df, threshold=.95, n_jobs=-1, dataset_name=None):
    df = df.copy()
    df['poem_text_no_diacritics'] = df['poem_text_no_diacritics'].fillna('')

    if dataset_name:
        cached = load_hash_cache(dataset_name)
        if cached:
            df['norm_text'], df['bucket'] = cached
        else:
            df['norm_text'] = df['poem_text_no_diacritics'].apply(normalize_text)
            df['bucket'] = df['norm_text'].apply(lambda x: hash_text(x))
            save_hash_cache(dataset_name, df['norm_text'], df['bucket'])
    else:
        df['norm_text'] = df['poem_text_no_diacritics'].apply(normalize_text)
        df['bucket'] = df['norm_text'].apply(lambda x: hash_text(x))

    non_missing_counts = df.notna().sum(axis=1).values
    grouped = df.groupby('bucket')
    all_blocks = [(group['poem_text_no_diacritics'].tolist(), group.index.tolist()) for _, group in grouped]

    to_drop = set()

    with tqdm(total=len(all_blocks), desc="Deduplicating") as pbar:
        def wrapper(block):
            return compare_block(*block, threshold, non_missing_counts, log_matches=True)

        for i in range(0, len(all_blocks), 1000):
            batch = all_blocks[i:i+1000]
            results = Parallel(n_jobs=n_jobs)(
                delayed(wrapper)(block) for block in batch
            )
            to_drop.update(*results)
            pbar.update(len(batch))

    return df.drop(index=list(to_drop)).reset_index(drop=True)


In [46]:
df_sample = df.sample(n=20000)

for threshold in [0.70, 0.90, 0.93, 0.95, 0.98]:
    print(f"\n=== Testing threshold: {threshold} ===")
    df_dedup = block_and_deduplicate(df_sample, threshold=threshold, n_jobs=1, dataset_name=None)
    removed = len(df_sample) - len(df_dedup)
    print(f"Removed {removed} duplicates at threshold {threshold}")


=== Testing threshold: 0.7 ===


Deduplicating: 100%|██████████| 19955/19955 [00:00<00:00, 327889.24it/s]



[Score 96.80672268907563]
A: ولرب نازلة يضيق بها الفتى	ذرعا وعند الله منها المخرج
ضاقت فلما استحكمت حلقاتها	فرجت وكنت أظنها لا تفرج
أرخى ذوائبها وفك حبالها	من باسمه كل الخلائق تلهج
هو ملجأ المحتاج دون مذلة	فالحر بين يديه لا يتحرج
ولرب دعوة صادق في محنة	عادت ولطف الله فيها مدرج
ولسوف يكشف ما بنا من غمة	فليدعه من للحوائج أحوج
B: ولرب نازلة يضيق بها الفتى
	ذرعا وعند الله منها المخرج
	ضاقت فلما استحكمت حلقاتها
فرجت وكنت أظنها لا تفرج	أرخى ذوائبها وفك حبالها
من باسمه كل الخلائق تلهج	هو ملجأ المحتاج دون مذلة
فالحر بين يديه لا يتحرج	ولرب دعوة صادق في محنة
عادت ولطف الله فيها مدرج	ولسوف يكشف ما بنا من غمة
	فليدعه من للحوائج أحوج

[Score 100.0]
A: ترحلت عن بغداد أطيب منزل	وأبهى بلاد الله مرأى ومنظرا
وفارقت أقواما إذا ما ذكرتهم	ترقرق ماء العين ثم تحدرا
فكم من أديب في معانيه بارع	وأبلج في علم الشريعة أزهرا
أروح على برح الهموم وأغتدي	أكابد أحزانا يضيق بها الثرا
ولم أبك مربع العامرية باللوى	ولا رسم دار بالثنية مقفرا
ولكنني أبكي مقامي ببلدة	أأمل أن ألقى صديقا ولا أرى
B: ترحلت عن بغداد أطيب منزل	وأب

Deduplicating: 100%|██████████| 19955/19955 [00:00<00:00, 322479.65it/s]



[Score 96.80672268907563]
A: ولرب نازلة يضيق بها الفتى	ذرعا وعند الله منها المخرج
ضاقت فلما استحكمت حلقاتها	فرجت وكنت أظنها لا تفرج
أرخى ذوائبها وفك حبالها	من باسمه كل الخلائق تلهج
هو ملجأ المحتاج دون مذلة	فالحر بين يديه لا يتحرج
ولرب دعوة صادق في محنة	عادت ولطف الله فيها مدرج
ولسوف يكشف ما بنا من غمة	فليدعه من للحوائج أحوج
B: ولرب نازلة يضيق بها الفتى
	ذرعا وعند الله منها المخرج
	ضاقت فلما استحكمت حلقاتها
فرجت وكنت أظنها لا تفرج	أرخى ذوائبها وفك حبالها
من باسمه كل الخلائق تلهج	هو ملجأ المحتاج دون مذلة
فالحر بين يديه لا يتحرج	ولرب دعوة صادق في محنة
عادت ولطف الله فيها مدرج	ولسوف يكشف ما بنا من غمة
	فليدعه من للحوائج أحوج

[Score 100.0]
A: ترحلت عن بغداد أطيب منزل	وأبهى بلاد الله مرأى ومنظرا
وفارقت أقواما إذا ما ذكرتهم	ترقرق ماء العين ثم تحدرا
فكم من أديب في معانيه بارع	وأبلج في علم الشريعة أزهرا
أروح على برح الهموم وأغتدي	أكابد أحزانا يضيق بها الثرا
ولم أبك مربع العامرية باللوى	ولا رسم دار بالثنية مقفرا
ولكنني أبكي مقامي ببلدة	أأمل أن ألقى صديقا ولا أرى
B: ترحلت عن بغداد أطيب منزل	وأب

...


[Score 96.80672268907563]
A: ولرب نازلة يضيق بها الفتى	ذرعا وعند الله منها المخرج
ضاقت فلما استحكمت حلقاتها	فرجت وكنت أظنها لا تفرج
أرخى ذوائبها وفك حبالها	من باسمه كل الخلائق تلهج
هو ملجأ المحتاج دون مذلة	فالحر بين يديه لا يتحرج
ولرب دعوة صادق في محنة	عادت ولطف الله فيها مدرج
ولسوف يكشف ما بنا من غمة	فليدعه من للحوائج أحوج
B: ولرب نازلة يضيق بها الفتى
	ذرعا وعند الله منها المخرج
	ضاقت فلما استحكمت حلقاتها
فرجت وكنت أظنها لا تفرج	أرخى ذوائبها وفك حبالها
من باسمه كل الخلائق تلهج	هو ملجأ المحتاج دون مذلة
فالحر بين يديه لا يتحرج	ولرب دعوة صادق في محنة
عادت ولطف الله فيها مدرج	ولسوف يكشف ما بنا من غمة
	فليدعه من للحوائج أحوج

[Score 100.0]
A: ترحلت عن بغداد أطيب منزل	وأبهى بلاد الله مرأى ومنظرا
وفارقت أقواما إذا ما ذكرتهم	ترقرق ماء العين ثم تحدرا
فكم من أديب في معانيه بارع	وأبلج في علم الشريعة أزهرا
أروح على برح الهموم وأغتدي	أكابد أحزانا يضيق بها الثرا
ولم أبك مربع العامرية باللوى	ولا رسم دار بالثنية مقفرا
ولكنني أبكي مقامي ببلدة	أأمل أن ألقى صديقا ولا أرى
B: ترحلت عن بغداد أطيب منزل	وأب

Deduplicating: 100%|██████████| 19955/19955 [00:00<00:00, 326229.09it/s]



[Score 96.80672268907563]
A: ولرب نازلة يضيق بها الفتى	ذرعا وعند الله منها المخرج
ضاقت فلما استحكمت حلقاتها	فرجت وكنت أظنها لا تفرج
أرخى ذوائبها وفك حبالها	من باسمه كل الخلائق تلهج
هو ملجأ المحتاج دون مذلة	فالحر بين يديه لا يتحرج
ولرب دعوة صادق في محنة	عادت ولطف الله فيها مدرج
ولسوف يكشف ما بنا من غمة	فليدعه من للحوائج أحوج
B: ولرب نازلة يضيق بها الفتى
	ذرعا وعند الله منها المخرج
	ضاقت فلما استحكمت حلقاتها
فرجت وكنت أظنها لا تفرج	أرخى ذوائبها وفك حبالها
من باسمه كل الخلائق تلهج	هو ملجأ المحتاج دون مذلة
فالحر بين يديه لا يتحرج	ولرب دعوة صادق في محنة
عادت ولطف الله فيها مدرج	ولسوف يكشف ما بنا من غمة
	فليدعه من للحوائج أحوج

[Score 100.0]
A: ترحلت عن بغداد أطيب منزل	وأبهى بلاد الله مرأى ومنظرا
وفارقت أقواما إذا ما ذكرتهم	ترقرق ماء العين ثم تحدرا
فكم من أديب في معانيه بارع	وأبلج في علم الشريعة أزهرا
أروح على برح الهموم وأغتدي	أكابد أحزانا يضيق بها الثرا
ولم أبك مربع العامرية باللوى	ولا رسم دار بالثنية مقفرا
ولكنني أبكي مقامي ببلدة	أأمل أن ألقى صديقا ولا أرى
B: ترحلت عن بغداد أطيب منزل	وأب

Deduplicating: 100%|██████████| 19955/19955 [00:00<00:00, 319814.36it/s]


[Score 100.0]
A: ترحلت عن بغداد أطيب منزل	وأبهى بلاد الله مرأى ومنظرا
وفارقت أقواما إذا ما ذكرتهم	ترقرق ماء العين ثم تحدرا
فكم من أديب في معانيه بارع	وأبلج في علم الشريعة أزهرا
أروح على برح الهموم وأغتدي	أكابد أحزانا يضيق بها الثرا
ولم أبك مربع العامرية باللوى	ولا رسم دار بالثنية مقفرا
ولكنني أبكي مقامي ببلدة	أأمل أن ألقى صديقا ولا أرى
B: ترحلت عن بغداد أطيب منزل	وأبهى بلاد الله مرأى ومنظرا
وفارقت أقواما إذا ما ذكرتهم	ترقرق ماء العين ثم تحدرا
فكم من أديب في معانيه بارع	وأبلج في علم الشريعة أزهرا
أروح على برح الهموم وأغتدي	أكابد أحزانا يضيق بها الثرا
ولم أبك مربع العامرية باللوى	ولا رسم دار بالثنية مقفرا
ولكنني أبكي مقامي ببلدة	أأمل أن ألقى صديقا ولا أرى

[Score 99.3103448275862]
A: أنا أبو حية واسمي ودعان	لا ضرع طفل ولا عود فان
	كيف ترى ضربى رؤوس الأقران
B: أنا أبو حية واسمي ودعان	لا ضرع طفل ولا عود فان
كيف ترى ضربى رؤوس الأقران

[Score 100.0]
A: تصبته ريحا شمأل وجنوب	فقال العدى ما حظه بقريب
وشاقته أمواج هناك يزورها	فيبكي على صخر لها وكثيب
ونثر آمالا على كل شاطئ	وسار كسير القلب بين شع

In [1]:
import pandas as pd


pd.read_csv('/path/to/data/outputs/de_dupped_test_with_keywords.csv')


,dataset_name,era,genre,location,meter,overall_explanation,poem_id,poem_language,poem_text,poem_title,...,rhyme,source,tags,verses_explanation,poem_text_no_diacritics,poet_name_with_diacritics,norm_text,bucket,keywords,key_phrases
0,FannOrFlop,NaN,عتاب,NaN,بحر الخفيف,[{'explanation': 'يستهل الشاعر قصيدته بتوبيخ أ...,4,NaN,إِنَّ عَضــّيكَ يــا أَخــي بِــالمَلامِ\tلا ي...,إِنَّ عَضّيكَ يا أَخي بِالمَلامِ,...,NaN,https://arabic-poetry.net/poem/20188-إن-عضيك-ي...,['الخفيف' 'عتاب' 'العصر الحديث'],Verses 1-10:\nتتحدث هذه القصيدة عن لوعة الشاعر...,إن عضــيك يــا أخــي بــالملام\tلا يــؤدي لمثـ...,حافظ ابراهيم,أخــي أعــدل أن أنت إذا إن ال الأجـــــرام الأ...,7151c0,"['العتاب', 'الخيانة', 'المعاناة']","['غير راعي الذمام', 'بات بين الظنون والأوهام',..."
1,FannOrFlop,NaN,عتاب,NaN,بحر الرجز,[{'explanation': 'يصف الشاعر نفسه بأنه شخصٌ تع...,5,NaN,مِن واجِدٍ مُنَقِّرِ المَنامِ\tطَريدَ دَهرٍ جا...,مِن واجِدٍ مُنَقِّرِ المَنامِ,...,NaN,https://arabic-poetry.net/poem/20185-من-واجد-م...,['الرجز' 'عتاب' 'العصر الحديث'],Verses 1-12:\nتتحدث هذه القصيدة عن شاعرٍ مُتعب...,من واجد منقر المنام\tطريد دهر جائر الأحكام\nمش...,حافظ ابراهيم,أبي أتى أدعوكم أرق أزهى أقسموا أم أن إذا إليكم...,214948,"['الهموم', 'الشوق', 'الذكرى']","['طريد دهر جائر الأحكام', 'يقضوا دولة الظلام',..."
2,FannOrFlop,NaN,رثاء,NaN,بحر الكامل,[{'explanation': 'يبدأ الشاعر بوصف واقع مؤلم، ...,7,NaN,بَـدَأَ المَمـاتُ يَـدِبُّ في أَترابي\tوَبَــد...,بَدَأَ المَماتُ يَدِبُّ في أَترابي,...,NaN,https://arabic-poetry.net/poem/20351-بدأ-المما...,['الكامل' 'رثاء' 'العصر الحديث'],Verses 1-4:\nتتحدث القصيدة عن فراق الحبيب ومرو...,بـدأ الممـات يـدب في أترابي\tوبــدأت أعـرف وحش...,حافظ ابراهيم,آمــالي أترابي أعـرف إلفك الأحبـاب الأحبـاب ال...,f424cf,"['الممات', 'الأحباب', 'الأصحاب']","['بدأ الممات يدب في أترابي', 'وحشة الأحبـاب', ..."
3,FannOrFlop,NaN,مدح,NaN,بحر الخفيف,[{'explanation': 'يبدأ الشاعر بمدح مناسبة احتف...,9,NaN,إِنَّ يَــومَ اِحتِفـالِكُم زادَ حُسـناً\tوَجَ...,إِنَّ يَومَ اِحتِفالِكُم زادَ حُسناً,...,NaN,https://arabic-poetry.net/poem/20249-إن-يوم-اح...,['الخفيف' 'مدح' 'العصر الحديث'],Verses 1-12:\nتتحدث القصيدة عن احتفالٍ مزدوج، ...,إن يــوم احتفـالكم زاد حسـنا\tوجلالا بيـــوم ع...,حافظ ابراهيم,آفـة آنسـوا أشــيم أظلـم أعضـائكم أغنـى أكمــه...,fe9cc6,NaN,NaN
4,FannOrFlop,NaN,عتاب,NaN,بحر البسيط,[{'explanation': 'يبدأ الشاعر بوصف سمر طال (حد...,11,NaN,طـالَ الحَديثُ عَلَيكُم أَيُّها السَمَرُ\tوَلا...,طالَ الحَديثُ عَلَيكُم أَيُّها السَمَرُ,...,NaN,https://arabic-poetry.net/poem/20180-طال-الحدي...,['البسيط' 'عتاب' 'العصر الحديث'],Verses 1-12:\nتتحدث القصيدة عن شاعر يعاني من ق...,طـال الحديث عليكم أيها السمر\tولاح للنـوم فـي ...,حافظ ابراهيم,آبسـة أبيـت أتنسى أتيح أثـر أجفـانكم أحشـاه أس...,156098,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3862,FannOrFlop,NaN,حكمه,NaN,بحر الرجز,[{'explanation': 'يؤكد هذا البيت على أن الخير ...,6911,NaN,الخَيـرُ خَيـرٌ كَاسـمِهِ\tوَالشــَرُّ شـَرٌّ ...,الخَيرُ خَيرٌ كَاسمِهِ,...,NaN,https://arabic-poetry.net/poem/2946-الخير-خير-...,['الرجز' 'حكمه' 'العصر العباسي'],Verses 1-5:\nتتحدث هذه القصيدة عن الخير والشر،...,الخيـر خيـر كاسـمه\tوالشــر شـر كاسـمه\nسبحان ...,أبو العَتاهِيَة,أرضـاه أسعد الخيـر العبا الله امرأ بسـابق بعـد...,f70dd1,NaN,NaN
3863,FannOrFlop,NaN,عتاب,NaN,بحر الرمل,[{'explanation': 'يا من تعاتبني (يا عاذلي)، اط...,6912,NaN,عـاذِلي فيها أَطِعني\tوَأَقِــلَّ الآنَ لَــوم...,عاذِلي فيها أَطِعني,...,NaN,https://arabic-poetry.net/poem/4109-عاذلي-فيها...,['الرمل' 'عتاب' 'العصر العباسي'],Verses 1-5:\nهذه قصيدة ذات طابع انعتاقي، يدعو ...,عـاذلي فيها أطعني\tوأقــل الآن لــومي\nواشرب ا...,أبو نُوّاس,أبـدا أطعني أو الآن الخمر الراح الصوم بشرب بعـ...,6d105a,NaN,NaN
3864,FannOrFlop,NaN,غزل,NaN,بحر الوافر,[{'explanation': 'يبدأ الشاعر بدعوة إلى التمتع...,6914,NaN,خـذِ العيـشَ الهنيَّ من المجوس\tمعــاقرةَ العق...,خذِ العيشَ الهنيَّ من المجوس,...,NaN,https://arabic-poetry.net/poem/4474-خذ-العيش-ا...,['الوافر' 'غزل' 'العصر العباسي'],Verses 1-12:\nتتحدث هذه القصيدة عن متعة الشراب...,خـذ العيـش الهني م